In [1]:
import pandas as pd
import numpy as np
import os

data_path = r"C:\Users\likhi\OneDrive\Desktop\e commerce project\data\\"
output_path = r"C:\Users\likhi\OneDrive\Desktop\e commerce project\outputs\\"

orders       = pd.read_csv(data_path + "olist_orders_dataset.csv", encoding='utf-8', low_memory=False)
order_items  = pd.read_csv(data_path + "olist_order_items_dataset.csv", encoding='utf-8', low_memory=False)
customers    = pd.read_csv(data_path + "olist_customers_dataset.csv", encoding='utf-8', low_memory=False)
products     = pd.read_csv(data_path + "olist_products_dataset.csv", encoding='utf-8', low_memory=False)
sellers      = pd.read_csv(data_path + "olist_sellers_dataset.csv", encoding='utf-8', low_memory=False)
reviews      = pd.read_csv(data_path + "olist_order_reviews_dataset.csv", encoding='utf-8', low_memory=False)
payments     = pd.read_csv(data_path + "olist_order_payments_dataset.csv", encoding='utf-8', low_memory=False)
category_translation = pd.read_csv(data_path + "product_category_name_translation.csv", encoding='utf-8', low_memory=False)

print("All tables loaded successfully")

All tables loaded successfully


In [2]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [3]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['late_delivery'] = (
    orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']
)

orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

print(orders[['order_id', 'delivery_days', 'late_delivery', 'order_month']].head(5))

                           order_id  delivery_days  late_delivery order_month
0  e481f51cbdc54678b7cc49136f2d6af7            8.0          False     2017-10
1  53cdb2fc8bc7dce0b6741e2150273451           13.0          False     2018-07
2  47770eb9100c2d0c44946d9cf07ec65d            9.0          False     2018-08
3  949d5b44dbf5de918fe9c16f97b45f8a           13.0          False     2017-11
4  ad21c59c0840e6cb83a9ceb5573f8159            2.0          False     2018-02


In [4]:
print(orders['late_delivery'].value_counts())
print(f"\nLate delivery rate: {orders['late_delivery'].mean()*100:.1f}%")

late_delivery
False    91614
True      7827
Name: count, dtype: int64

Late delivery rate: 7.9%


In [5]:
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

print(f"All orders: {len(orders)}")
print(f"Delivered orders: {len(delivered_orders)}")
print(f"Dropped: {len(orders) - len(delivered_orders)} orders")

All orders: 99441
Delivered orders: 96478
Dropped: 2963 orders


In [6]:
df = delivered_orders.merge(order_items, on='order_id', how='left')
print(f"After merging order_items: {df.shape}")

df = df.merge(products, on='product_id', how='left')
print(f"After merging products: {df.shape}")

df = df.merge(category_translation, on='product_category_name', how='left')
print(f"After merging category_translation: {df.shape}")

df = df.merge(customers, on='customer_id', how='left')
print(f"After merging customers: {df.shape}")

df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')
print(f"After merging reviews: {df.shape}")

After merging order_items: (110197, 17)
After merging products: (110197, 25)
After merging category_translation: (110197, 26)
After merging customers: (110197, 30)
After merging reviews: (110840, 31)


In [7]:
columns_to_keep = [
    'order_id', 'order_status', 'order_purchase_timestamp',
    'order_estimated_delivery_date', 'order_delivered_customer_date',
    'delivery_days', 'late_delivery', 'order_month',
    'price', 'freight_value', 'product_category_name_english',
    'seller_id', 'customer_state', 'review_score'
]

df_clean = df[columns_to_keep].copy()
print(f"Columns reduced to: {df_clean.shape[1]}")

Columns reduced to: 14


In [8]:
df_clean = df_clean.dropna(subset=['order_delivered_customer_date'])
df_clean['product_category_name_english'] = df_clean['product_category_name_english'].fillna('unknown')

print(df_clean.isnull().sum())
print(f"\nFinal shape: {df_clean.shape}")

df_clean.to_csv(output_path + "clean_orders.csv", index=False, encoding='utf-8')
print("\nclean_orders.csv saved successfully!")

order_id                           0
order_status                       0
order_purchase_timestamp           0
order_estimated_delivery_date      0
order_delivered_customer_date      0
delivery_days                      0
late_delivery                      0
order_month                        0
price                              0
freight_value                      0
product_category_name_english      0
seller_id                          0
customer_state                     0
review_score                     827
dtype: int64

Final shape: (110832, 14)

clean_orders.csv saved successfully!


In [9]:
df_clean = pd.read_csv(output_path + "clean_orders.csv")

# Create a proper month_year text column
df_clean['month_year'] = pd.to_datetime(df_clean['order_purchase_timestamp']).dt.strftime('%Y-%m')

# Save back
df_clean.to_csv(output_path + "clean_orders.csv", index=False, encoding='utf-8')

print(df_clean['month_year'].value_counts().sort_index().head(10))

month_year
2016-09       3
2016-10     317
2016-12       1
2017-01     924
2017-02    1869
2017-03    2917
2017-04    2577
2017-05    4047
2017-06    3524
2017-07    4460
Name: count, dtype: int64


In [10]:
df = pd.read_csv(output_path + "clean_orders.csv")
print(f"Avg order value: R${df['price'].mean():.0f}")
print(f"Total revenue: R${df['price'].sum():,.0f}")

Avg order value: R$120
Total revenue: R$13,278,587
